In [ ]:
import tensorflow as tf
import os

url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
path = tf.keras.utils.get_file("flower_photos", url, untar=True)

# arahkan ke folder yang berisi subfolder kelas
data_dir = path
if not os.path.isdir(os.path.join(path, "daisy")):
    data_dir = os.path.join(path, "flower_photos")
print("Folder data:", data_dir, os.listdir(data_dir))

train = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=42,
    image_size=(224, 224), batch_size=32)
val = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=(224, 224), batch_size=32)

kelas = train.class_names
print(kelas)

Folder data: /root/.keras/datasets/flower_photos/flower_photos ['sunflowers', 'dandelion', 'roses', 'tulips', 'LICENSE.txt', 'daisy']
Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.
['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


In [ ]:
from tensorflow.keras import layers, models

base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet")
base.trainable = False

model = models.Sequential([
    layers.Input((224, 224, 3)),
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(kelas), activation="softmax"),
])

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train, validation_data=val, epochs=5)

Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 159s 2s/step - accuracy: 0.7585 - loss: 0.7732 - val_accuracy: 0.8597 - val_loss: 0.4013
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 137s 1s/step - accuracy: 0.8716 - loss: 0.3743 - val_accuracy: 0.8924 - val_loss: 0.3124
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 138s 2s/step - accuracy: 0.8954 - loss: 0.2944 - val_accuracy: 0.9033 - val_loss: 0.3211
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 151s 2s/step - accuracy: 0.9169 - loss: 0.2267 - val_accuracy: 0.9005 - val_loss: 0.3047
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 141s 2s/step - accuracy: 0.9268 - loss: 0.1915 - val_accuracy: 0.9033 - val_loss: 0.3284


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import pandas as pd

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train.prefetch(AUTOTUNE)
val_ds = val.prefetch(AUTOTUNE)

augment = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def buat_model(lr=1e-3, dropout=0.3, units=128):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3), include_top=False, weights="imagenet")
    base.trainable = False
    m = models.Sequential([
        layers.Input((224, 224, 3)),
        augment,  # hanya aktif saat training
        layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
        base,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(units, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(len(kelas), activation="softmax"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m, base


In [ ]:
konfigurasi = [
    {"lr": 1e-3, "dropout": 0.3, "units": 128},   # baseline
    {"lr": 1e-4, "dropout": 0.3, "units": 128},
    {"lr": 1e-3, "dropout": 0.5, "units": 128},
    {"lr": 1e-3, "dropout": 0.3, "units": 256},
]

hasil = []
terbaik = None
for cfg in konfigurasi:
    print("Mencoba:", cfg)
    m, b = buat_model(**cfg)
    h = m.fit(train_ds, validation_data=val_ds, epochs=5, verbose=2)
    va = max(h.history["val_accuracy"])
    hasil.append({**cfg,
                  "train_acc": h.history["accuracy"][-1],
                  "val_acc": va,
                  "val_loss": min(h.history["val_loss"])})
    if terbaik is None or va > terbaik[0]:
        terbaik = (va, cfg)

tabel = pd.DataFrame(hasil).sort_values("val_acc", ascending=False)
print(tabel)          # <- screenshot / salin ke laporan
print("Konfigurasi terbaik:", terbaik[1])


In [ ]:
zmodel, base = buat_model(**terbaik[1])
h1 = model.fit(train_ds, validation_data=val_ds, epochs=8, verbose=2)

# Fine-tuning: buka 30 layer teratas MobileNetV2 (BatchNorm tetap beku)
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=6, verbose=2)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# kurva akurasi & loss (gabungan tahap 1 + fine-tuning)
acc = h1.history["accuracy"] + h2.history["accuracy"]
vacc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
los = h1.history["loss"] + h2.history["loss"]
vlos = h1.history["val_loss"] + h2.history["val_loss"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(acc, label="train"); ax[0].plot(vacc, label="val")
ax[0].axvline(len(h1.history["accuracy"]) - 1, ls="--", c="gray")
ax[0].set_title("Akurasi"); ax[0].legend()
ax[1].plot(los, label="train"); ax[1].plot(vlos, label="val")
ax[1].axvline(len(h1.history["loss"]) - 1, ls="--", c="gray")
ax[1].set_title("Loss"); ax[1].legend()
plt.savefig("kurva_training.png", dpi=150, bbox_inches="tight")
plt.show()

# prediksi dan label diambil dalam satu lintasan (val di-shuffle)
y_true, y_pred = [], []
for xb, yb in val:
    p = model.predict(xb, verbose=0)
    y_true.extend(yb.numpy()); y_pred.extend(np.argmax(p, axis=1))

print(classification_report(y_true, y_pred, target_names=kelas, digits=4))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=kelas).plot(xticks_rotation=45)
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
model.save("model_cnn.keras")
import json
json.dump(list(kelas), open("kelas.json", "w"))
from google.colab import files
files.download("model_cnn.keras")
files.download("kurva_training.png")
files.download("confusion_matrix.png")
